In [32]:
import cloudComPy as ccp
import numpy as np
data = 'gw.bin'
fullcloud = ccp.loadPointCloud(data)
xyz = fullcloud.toNpArray()

curv_radii = np.arange(0.5,1.6,0.5)
dens3d_radii = np.arange(1.05,0.15,-0.3)
print(curv_radii, dens3d_radii)
assert len(curv_radii) == len(dens3d_radii), 'need to have the same number of curvature and density radii'

curv = ccp.CurvatureType.GAUSSIAN_CURV
dens3d = ccp.Density.DENSITY_3D

test_cloud = ccp.ccPointCloud()
test_point = [-66.638,-48.05,1840.893]
box_size = 50
#create a box around the test point and extract the points within that box

box_min = np.array(test_point) - box_size/2
box_max = np.array(test_point) + box_size/2
print(box_min, box_max)
box_mask = np.all((xyz >= box_min) & (xyz <= box_max), axis=1)
box_points = xyz[box_mask]
print(box_points.shape)
print(np.min(box_points, axis=0), np.max(box_points, axis=0))
test_cloud.coordsFromNPArray_copy(box_points)

# compute the features for the test cloud
for sr,dr in zip(curv_radii, dens3d_radii):
    ccp.computeLocalDensity(dens3d, dr, [test_cloud])
    ccp.computeCurvature(curv, sr, [test_cloud])

names = test_cloud.getScalarFieldDic()
curv_names = [n for n in names if 'curvature' in n]
dens_names = [n for n in names if 'density' in n]
# sort density in opposite order of curvature based on the radii used to compute them
# the number is in ()
curv_names = sorted(curv_names, key=lambda x: float(x.split('(')[1].split(')')[0]), reverse=True)
dens_names = sorted(dens_names, key=lambda x: float(x.split('=')[1].split(')')[0]))
print('curvature names:', curv_names)
print('density names:', dens_names)

ratios = []
for cn, dn in zip(curv_names, dens_names):
    curv_values = test_cloud.getScalarField(names[cn]).toNpArray()
    dens_values = test_cloud.getScalarField(names[dn]).toNpArray()
    #normalize the curvature and density values to be between 0 and 1
    curv_values = (curv_values - np.nanmin(curv_values)) / (np.nanmax(curv_values) - np.nanmin(curv_values))
    dens_values = (dens_values - np.nanmin(dens_values)) / (np.nanmax(dens_values) - np.nanmin(dens_values))
    ratio = curv_values - dens_values
    ratios.append(ratio)
    print(f'{cn} / {dn} ratio: {ratio}')

total = np.sum(ratios, axis=0)
test_cloud.addScalarField('curv_density_ratio')
sf = test_cloud.getScalarField('curv_density_ratio')
sf.fromNpArrayCopy(total)
names = test_cloud.getScalarFieldDic()

test_cloud.sfBilateralFilter(names['curv_density_ratio'])

ccp.SavePointCloud(test_cloud, 'test_output.bin')



[0.5 1.  1.5] [1.05 0.75 0.45]
[ -91.638  -73.05  1815.893] [ -41.638  -23.05  1865.893]
(5252325, 3)
[ -91.58221  -73.04999 1827.2902 ] [ -52.04238  -23.05001 1865.893  ]
curvature names: ['Gaussian curvature (1.5)', 'Gaussian curvature (1)', 'Gaussian curvature (0.5)']
density names: ['Volume density (r=0.45)', 'Volume density (r=0.75)', 'Volume density (r=1.05)']
Gaussian curvature (1.5) / Volume density (r=0.45) ratio: [-0.23195896 -0.24808395 -0.23125038 ... -0.43517113 -0.35719424
 -0.4916655 ]
Gaussian curvature (1) / Volume density (r=0.75) ratio: [-0.20276177 -0.24258396 -0.20168304 ... -0.5537712  -0.3913944
 -0.6801433 ]
Gaussian curvature (0.5) / Volume density (r=1.05) ratio: [-0.17841189 -0.19349596 -0.18086457 ... -0.55172455 -0.36956885
 -0.7559765 ]


<CC_FILE_ERROR.CC_FERR_NO_ERROR: 0>

In [29]:
names

{'Gaussian curvature (0.5)': 1,
 'Gaussian curvature (1)': 3,
 'Gaussian curvature (1.5)': 5,
 'Volume density (r=0.45)': 4,
 'Volume density (r=0.75)': 2,
 'Volume density (r=1.05)': 0}